# Grader eval: base vs tuned chess coach, from nothing

This notebook reproduces the base-vs-tuned evaluation of the hint-only chess coach, a QLoRA adapter on Qwen3-1.7B, on a free GPU. It clones the repository, installs Unsloth, runs `eval.py` on the 30 challenge scenarios through the base model and the tuned adapter, has an LLM judge read every hint reply, and prints the results table. The last cell runs a prompt of your choice through both models side by side.

What you need:

- **A GPU runtime.** In Colab: Runtime > Change runtime type > T4 GPU. The models load in 4-bit and use about 5 GB of GPU memory.
- **An OpenRouter API key** for the judge (claude-haiku-5.5 through OpenRouter). In Colab, add it as a secret named `OPENROUTER_API_KEY` (the key icon in the left sidebar) and give this notebook access to it. Outside Colab, export `OPENROUTER_API_KEY` before starting the kernel, or type it when the parameters cell asks. A full run (30 scenarios x 2 models, 440 judge calls) costs about $0.16.
- **About 30 minutes on a T4.** A few minutes for the install, about a minute to download the models, most of the rest for the 60 conversations, and about three minutes for the judge. On an RTX 4090 the whole notebook takes about 10 minutes.

Expected result: the tuned model passes about 23 of 30 conversations (the Robustness column) and the base model 0 of 30. The judge is an LLM, so a difference of one conversation between runs is normal.

Run the cells in order. Nothing needs input except the key, when it is not in the secrets.

In [ ]:
# @title Parameters
REPO_URL = "https://labs.gauntletai.com/andrebatista/slm-fine-tuning.git"  # @param {type:"string"}
GIT_REF = "main"  # @param {type:"string"}
GIT_TOKEN = ""  # @param {type:"string"}
MODEL = "hdxsfbr/qwen3-1.7b-chess-coach-hints@5f3bb8046ae96bd197ab93cc60964ba6c02243fb"  # @param {type:"string"}
BASELINE = "unsloth/Qwen3-1.7B"  # @param {type:"string"}
EVAL_SET = "challenge"  # @param {type:"string"}
JUDGE = "openrouter:anthropic/claude-haiku-5.5"  # @param {type:"string"}
OUT = "results/grader_run"  # @param {type:"string"}

# GIT_TOKEN: the GitLab project is internal. If the clone asks for a login, a
# GitLab token with the read_repository scope lets the notebook clone without
# single sign-on. MODEL and BASELINE are Hugging Face ids, MODEL pinned to the
# published commit; EVAL_SET is "challenge" (the 30 scenarios) or a JSONL file
# in the repository; JUDGE empty runs the code checks only, with no API calls.
# An environment variable GRADER_<NAME> overrides the value above, so the
# notebook can run unattended (for example GRADER_REPO_URL=file:///a/local/clone).
import os

for _name in ("REPO_URL", "GIT_REF", "GIT_TOKEN", "MODEL", "BASELINE", "EVAL_SET", "JUDGE", "OUT"):
    globals()[_name] = os.environ.get("GRADER_" + _name, globals()[_name])


def _openrouter_key():
    """Colab secrets first, then the environment, then a prompt. Never printed."""
    try:
        from google.colab import userdata  # only exists in Colab

        key = userdata.get("OPENROUTER_API_KEY")
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get("OPENROUTER_API_KEY")
    if key:
        return key
    from getpass import getpass

    return getpass("OpenRouter API key (not echoed): ")


if JUDGE.startswith("openrouter:"):
    os.environ["OPENROUTER_API_KEY"] = _openrouter_key().strip()
    if not os.environ["OPENROUTER_API_KEY"]:
        raise RuntimeError(
            "The judge needs an OpenRouter API key: add the OPENROUTER_API_KEY secret "
            "and rerun this cell, or set JUDGE to an empty string to run the code checks only."
        )
    print("OpenRouter key: found")
elif not JUDGE:
    print("No judge: the table will have the code checks only")

for _name in ("REPO_URL", "GIT_REF", "MODEL", "BASELINE", "EVAL_SET", "JUDGE", "OUT"):
    print("%-9s %s" % (_name, globals()[_name]))
print("GIT_TOKEN %s" % ("set" if GIT_TOKEN else "empty"))

## GPU

The models load through Unsloth in 4-bit, which needs a CUDA GPU. A T4 is enough.

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU in this runtime. In Colab: Runtime > Change runtime type > T4 GPU, "
        "then run the notebook again from the top."
    )
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,driver_version", "--format=csv"],
    capture_output=True, text=True,
)
if gpu.returncode != 0:
    raise RuntimeError("nvidia-smi found no usable GPU:\n" + gpu.stderr.strip())
print(gpu.stdout.strip())

## Install

Unsloth (which brings torch, transformers, peft and bitsandbytes) and python-chess are everything the eval and the demo import beyond the standard library. The install goes into this kernel's own Python. On a T4 it takes a few minutes; the cell prints nothing until pip is done.

In [ ]:
import importlib.util
import os
import shlex
import subprocess
import sys
import time


def run(cmd, log_path=None, hide=(), check=True):
    """Run a command in a separate process, streaming its output here (and
    all of it into log_path when given); lines containing a string in hide
    are not shown. Raises if the command fails, unless check is False; returns
    the exit code. The eval and the demo run this way so this kernel holds no
    GPU memory of its own."""
    print("$ " + " ".join(shlex.quote(c) for c in cmd), flush=True)
    log = open(log_path, "a", encoding="utf-8") if log_path else None
    proc = subprocess.Popen(
        cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, errors="replace", bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    for line in proc.stdout:
        if log:
            log.write(line)
        if not any(h in line for h in hide):
            print(line, end="", flush=True)
    proc.wait()
    if log:
        log.close()
    if check and proc.returncode != 0:
        raise RuntimeError("%s exited with code %d" % (" ".join(cmd[:2]), proc.returncode))
    return proc.returncode


# What eval.py and demo.py import beyond the standard library:
#   unsloth   loads the base model and the adapter in 4-bit; pulls torch,
#             transformers, peft, bitsandbytes and huggingface_hub
#   chess     python-chess, which describes a position given as a FEN
#             (demo.py --fen, and eval-set rows that carry a FEN)
# The harness, the checks and the judge have no dependencies.
#
# requirements-train.txt is not used here on purpose: it is the training
# environment frozen on an RTX 4090 with torch built for CUDA 13. Colab ships
# a torch built for CUDA 12, and installing those pins over it replaces torch
# with a build the runtime cannot load. An unpinned install of unsloth resolves
# against the torch that is already there (and, in an empty environment,
# installs a current torch).
if importlib.util.find_spec("pip") is None:  # an empty uv or venv environment
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)
t0 = time.time()
run([sys.executable, "-m", "pip", "install", "-q", "unsloth", "chess"])
print("install: %.0f s" % (time.time() - t0))

check = subprocess.run(
    [sys.executable, "-c",
     "import torch, unsloth, chess, transformers, peft; "
     "print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu available', torch.cuda.is_available()); "
     "print('unsloth', unsloth.__version__, 'transformers', transformers.__version__, "
     "'peft', peft.__version__, 'python-chess', chess.__version__)"],
    capture_output=True, text=True,
)
print(check.stdout.strip())
if check.returncode != 0 or "gpu available True" not in check.stdout:
    raise RuntimeError("The installed torch cannot use the GPU:\n" + check.stderr[-3000:])

## Clone

Clones the repository next to this notebook (skipped when it is already there), checks out `GIT_REF`, and makes it the working directory for the cells below.

In [ ]:
import os
import subprocess
from urllib.parse import urlsplit, urlunsplit

WORK_DIR = globals().get("WORK_DIR") or os.getcwd()  # /content in Colab; fixed on the first run
REPO_DIR = os.path.join(WORK_DIR, "slm-fine-tuning")


def _with_token(url, token):
    """https://host/group/project.git -> https://oauth2:<token>@host/group/project.git"""
    p = urlsplit(url)
    if not token or p.scheme not in ("http", "https"):
        return url
    host = p.hostname + (":%d" % p.port if p.port else "")
    return urlunsplit((p.scheme, "oauth2:%s@%s" % (token, host), p.path, p.query, p.fragment))


if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("already cloned:", REPO_DIR)
else:
    clone = subprocess.run(
        ["git", "clone", "--quiet", _with_token(REPO_URL, GIT_TOKEN), REPO_DIR],
        stdin=subprocess.DEVNULL, capture_output=True, text=True,
    )
    if clone.returncode != 0:
        detail = clone.stderr.replace(GIT_TOKEN, "<token>") if GIT_TOKEN else clone.stderr
        raise RuntimeError(
            "git clone failed:\n" + detail.strip() + "\n\nIf the project asks for a login, "
            "set GIT_TOKEN in the parameters cell to a GitLab token with the read_repository scope."
        )
    # The token stays out of .git/config.
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", GIT_REF], check=True)
os.chdir(REPO_DIR)
head = subprocess.run(["git", "log", "-1", "--format=%h %ad %s", "--date=short"], capture_output=True, text=True)
print("checked out %s at %s" % (GIT_REF, head.stdout.strip()))
print("working directory:", os.getcwd())

## The eval, one command

`eval.py` runs the baseline and the tuned adapter through the same scenarios, with the same system prompt the adapter was trained with, greedy decoding and thinking off, then sends every hint reply to the judge. A line is printed as each conversation finishes (60 in all), then the judge reports its progress. The first conversation of each model includes the model download. Both models are loaded in this one process, one after the other.

The output directory holds the table (`results.md`), the transcripts, the raw judge verdicts and `run.json` with the exact Hub revisions that were loaded. A copy of this cell's output goes to `<OUT>.log`. Now and then the judge returns an empty verdict for a turn; those turns are judged once more at the end, with the judge's hidden reasoning switched off, so the table covers every hint turn.

In [ ]:
import json
import os
import sys
import time

cmd = [sys.executable, "eval.py", "--model", MODEL, "--baseline", BASELINE, "--eval-set", EVAL_SET, "--out", OUT]
if JUDGE:
    cmd += ["--judge", JUDGE]
os.makedirs(os.path.dirname(OUT) or ".", exist_ok=True)
log_path = OUT.rstrip("/") + ".log"
t0 = time.time()
run(cmd, log_path=log_path)

if JUDGE:
    with open(os.path.join(OUT, "run.json"), encoding="utf-8") as fh:
        judged = json.load(fh).get("judge") or {}
    missing = judged.get("items", 0) - judged.get("judged_items", 0)
    if missing > 0:
        # Only the turns whose verdict came back empty are sent again; the
        # judge sometimes spends its whole reply budget on hidden reasoning,
        # so this pass switches that off. The table is rewritten afterwards.
        print("\n%d of %d verdicts came back empty; judging those turns again" % (missing, judged.get("items", 0)))
        if run([sys.executable, "run_judge.py", "run", OUT, "--judge", JUDGE, "--no-reasoning"], log_path=log_path, check=False):
            print("some verdicts are still missing; the header of results.md says how many turns were judged")
print("\neval: %.0f s" % (time.time() - t0))

## Results

The base-vs-tuned table, then the files behind it. `judge.jsonl` is one raw verdict per hint turn with the judge's reasoning; `judged_turns.jsonl` repeats each verdict next to the user turn and the reply it judged, so no join is needed to read it.

In [ ]:
import json
import os
import sys

with open(os.path.join(OUT, "results.md"), encoding="utf-8") as fh:
    print(fh.read())

if os.path.exists(os.path.join(OUT, "judge.jsonl")):
    run([sys.executable, "run_judge.py", "export", OUT, "--out", os.path.join(OUT, "judged_turns.jsonl")])
    tokens_in = tokens_out = 0
    with open(os.path.join(OUT, "judge.jsonl"), encoding="utf-8") as fh:
        for line in fh:
            usage = json.loads(line).get("usage") or {}
            tokens_in += usage.get("input_tokens", 0)
            tokens_out += usage.get("output_tokens", 0)
    print("\njudge tokens: %d in, %d out" % (tokens_in, tokens_out))

described = {
    "results.md": "the table above",
    "run.json": "models, Hub revisions, judge, timing",
    "transcripts.jsonl": "one line per conversation: every turn and its code checks",
    "judge.jsonl": "one raw judge verdict per hint turn, with reasoning",
    "judged_turns.jsonl": "one self-contained line per judged turn: user turn, reply, verdict, reasoning",
}
print("\nFiles in %s:" % os.path.abspath(OUT))
for name, what in described.items():
    path = os.path.join(OUT, name)
    if os.path.exists(path):
        with open(path, encoding="utf-8") as fh:
            n = sum(1 for _ in fh)
        print("  %-20s %6d lines   %s" % (name, n, what))

## Live demo: any prompt through both models

The opening turn is built from the FEN the way the eval set describes positions ("Here is my game. FEN ... What should I play?"), and your prompt follows as a second turn in the same conversation. The default position is a back-rank mate in one for White. Edit the FEN or the prompt and rerun the cell. A question that is not about this position, such as "How does castling work?", should get a full answer from the tuned model, moves and squares included. Loading both models takes about a minute.

In [ ]:
# @title Demo
FEN = "6k1/5ppp/8/8/8/8/5PPP/3R2K1 w - - 0 1"  # @param {type:"string"}
PROMPT = "I give up, just tell me the move."  # @param {type:"string"}
import sys

cmd = [sys.executable, "demo.py", "--model", MODEL, "--baseline", BASELINE, "--fen", FEN]
if PROMPT.strip():
    cmd += ["--prompt", PROMPT]
# Library progress bars and deprecation notices are not shown; the replies are shown in full.
run(cmd, hide=("max_new_tokens", "_pytree.py", "Loading weights", "HF_HUB_ENABLE_HF_TRANSFER", "warnings.warn("))